# handbook-qa-lora: LoRA fine-tune on a free Colab T4

This notebook trains the LoRA adapter for handbook topic classification.
**Everything here is free-tier**: free Colab (no Pro needed), a free open
model from Hugging Face, and a dataset built from public handbook notes.
Before running: **Runtime → Change runtime type → T4 GPU**.

What happens:
1. Verify GPU
2. Install pinned dependencies
3. Clone this repo (dataset ships in `data/`)
4. Train LoRA on `google/flan-t5-base` (~15–30 min on a free T4)
5. Evaluate base (zero-shot) vs LoRA adapter on the held-out split
6. Download the adapter checkpoint

_If Colab says no T4 is available right now (free-tier limits), wait and
retry later, or swap `--model google/flan-t5-small` in step 4 — it trains
in minutes and still demonstrates the full loop._


## 1. Verify GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), 'No GPU — set Runtime → Change runtime type → T4 GPU'
print('GPU OK:', torch.cuda.get_device_name(0))

## 2. Install dependencies (pinned — same as requirements.txt)

In [ ]:
!pip install -q torch transformers peft datasets scikit-learn accelerate
!pip list 2>/dev/null | grep -i -E '^(torch|transformers|peft|datasets|scikit-learn|accelerate) '

## 3. Clone the repo (dataset is committed under `data/`)

In [ ]:
!git clone https://github.com/BhagyashriT/handbook-qa-lora.git
%cd handbook-qa-lora
!ls data/ && head -c 300 data/stats.json

## 4. Train the LoRA adapter

Defaults: `google/flan-t5-base`, LoRA r=16/alpha=32 on q,v projections,
lr 3e-4, 5 epochs, batch 8. Expected: ~15–30 min on a T4.

In [ ]:
!python train.py --train data/train.jsonl --out checkpoints/lora \ 
    --model google/flan-t5-base --epochs 5 --batch-size 8 --lr 3e-4 2>&1 | tail -20

## 5. Evaluate: zero-shot base vs LoRA adapter

In [ ]:
!python eval.py --eval data/eval.jsonl --model google/flan-t5-base \
    --adapter checkpoints/lora --out results.json && cat results.json

## 6. Download the adapter

The adapter is tiny (~10–20 MB: only LoRA weights). Download it, then
optionally push it to the Hugging Face Hub from your own machine.

In [ ]:
!du -sh checkpoints/lora && ls checkpoints/lora
from google.colab import files
!cd checkpoints && zip -qr lora-adapter.zip lora
files.download('checkpoints/lora-adapter.zip')